# 05 — Quantitative Uncertainty — Conformal Prediction
**Aviation Safety Risk Prediction — NTSB + NOAA Dataset**

### Goal
Wrap the best model from NB04 with **Conformal Prediction (CP)** to produce *guaranteed* coverage
prediction sets, rather than point predictions.

Instead of predicting "this accident will be FATL", the system outputs:
> "At 90% confidence, the injury class is in {FATL, SERS}"

This is crucial for aviation safety: the system tells you *when it is uncertain* and what the
possible outcomes are — not just the most likely one.

### Contents
1. Install & import (MAPIE)
2. Load best model + data
3. Calibrate conformal predictor on the validation set
4. Evaluate coverage at multiple α levels
5. Analyse prediction set sizes
6. Analyse when the model is uncertain
7. Distance to station vs uncertainty
8. Save conformal model and report

### Theoretical guarantee
For exchangeable data, MAPIE's RAPS method guarantees:
`P(y_true ∈ prediction_set) ≥ 1 − α`

Coverage is marginal (averaged over the test set), not conditional per sample.


In [ ]:
# Install MAPIE if not already installed
# pip install mapie
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', 'mapie', '-q'], check=False)


: 

In [ ]:
import importlib
import mapie
importlib.reload(mapie)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
from mapie.classification import MapieClassifier
from mapie.metrics import classification_coverage_score, classification_mean_width_score
from sklearn.metrics import (classification_report, f1_score, accuracy_score,confusion_matrix, roc_auc_score)

SEED       = 42
OUTPUT_DIR = 'outputs'
MODEL_DIR  = 'baseline_models'
UNCERT_DIR = 'uncertainty_outputs'
os.makedirs(UNCERT_DIR, exist_ok=True)

LABEL_MAP   = {0:'NONE', 1:'MINR', 2:'SERS', 3:'FATL'}
LABEL_NAMES = ['NONE', 'MINR', 'SERS', 'FATL']
CLASS_ORDER = [0, 1, 2, 3]
ALPHA_LEVELS = [0.05, 0.10, 0.15, 0.20]

np.random.seed(SEED)
print(f'MAPIE version: {mapie.__version__}')
print('Imports done ✓')

## 1. Load Best Model and Data

In [ ]:
import joblib, re, warnings
warnings.filterwarnings('ignore')

best_model = joblib.load(f'{MODEL_DIR}/best_model.pkl')
print(f'Best model type: {type(best_model).__name__}')

X_val_raw  = pd.read_csv(f'{OUTPUT_DIR}/X_val.csv')
X_test_raw = pd.read_csv(f'{OUTPUT_DIR}/X_test.csv')

y_val  = pd.read_csv(f'{OUTPUT_DIR}/y_val.csv').squeeze()
y_test = pd.read_csv(f'{OUTPUT_DIR}/y_test.csv').squeeze()

# LightGBM column sanitization (consistent with NB04)
def sanitize_lgb_columns(df):
    df = df.copy()
    new_cols = [re.sub(r'[^A-Za-z0-9_]', '_', col) for col in df.columns]
    seen = {}; deduped = []
    for col in new_cols:
        if col in seen: seen[col] += 1; deduped.append(f'{col}_{seen[col]}')
        else: seen[col] = 0; deduped.append(col)
    df.columns = deduped
    return df

is_lgb = 'LGBM' in type(best_model).__name__
if is_lgb:
    X_val  = sanitize_lgb_columns(X_val_raw)
    X_test = sanitize_lgb_columns(X_test_raw)
    print('LightGBM detected — using sanitized columns')
else:
    X_val  = X_val_raw.values
    X_test = X_test_raw.values

print(f'X_val  shape: {X_val.shape}')
print(f'X_test shape: {X_test.shape}')

try:
    feat_names = pd.read_csv(f'{OUTPUT_DIR}/feature_names.csv').squeeze().tolist()
except:
    feat_names = [f'f{i}' for i in range(X_val.shape[1])]


In [ ]:
# Baseline point-prediction performance reminder
y_pred_val = best_model.predict(X_val)
print('Point prediction performance on val set (pre-conformal):')
print(classification_report(y_val, y_pred_val, target_names=LABEL_NAMES, zero_division=0))


## 2. Conformal Prediction — Theory
**Split conformal prediction** (also called *inductive CP*):
1. Use the *calibration set* (here: the validation set) to compute non-conformity scores
2. For a new sample at significance level α, include class k in the prediction set if its
   non-conformity score is below the (1−α) quantile of calibration scores

**MAPIE** implements this with `cv='prefit'`, meaning the base model is already fitted
and only calibration is performed.

We use the **RAPS** (Regularized Adaptive Prediction Sets) method which tends to produce
more informative (tighter) prediction sets than naive softmax thresholding.


In [ ]:
# Calibrate LAC on the validation set
mapie = MapieClassifier(
    estimator=best_model,
    method='lac',
    cv='prefit',
    random_state=SEED,
)
mapie.fit(X_val, y_val)
mapie_90 = mapie
print('LAC calibrated on validation set ✓')
print(f'  Calibration samples: {len(y_val):,}')

## 3. Coverage Evaluation at Multiple α Levels

In [ ]:
coverage_results = []

for alpha in ALPHA_LEVELS:
    m = MapieClassifier(
        estimator=best_model,
        ,
        
        #method='lac',
        random_state=SEED,
    )
    m.fit(X_val, y_val)
    _, pred_sets_raw = m.predict_set(X_test, alpha=alpha)
    pred_sets = pred_sets_raw[:, :, 0]

    empirical_coverage = classification_coverage_score(y_test.values, pred_sets)
    mean_set_size      = classification_mean_width_score(pred_sets)
    set_sizes          = pred_sets.sum(axis=1)
    frac_singleton     = (set_sizes == 1).mean()
    frac_full_set      = (set_sizes == 4).mean()

    coverage_results.append({
        'alpha': alpha,
        'target_coverage': 1 - alpha,
        'empirical_coverage': float(empirical_coverage),
        'mean_set_size':      float(mean_set_size),
        'frac_singleton':     float(frac_singleton),
        'frac_full_set':      float(frac_full_set),
    })
    print(f"  α={alpha:.2f}  target={(1-alpha)*100:.0f}%  "
          f"empirical={float(empirical_coverage):.3f}  "
          f"mean_set_size={float(mean_set_size):.2f}  "
          f"singletons={float(frac_singleton)*100:.1f}%")

import pandas as pd
cov_df = pd.DataFrame(coverage_results)
cov_df.to_csv(f'{UNCERT_DIR}/coverage_results.csv', index=False)
print('\nCoverage results saved ✓')

In [ ]:
# Coverage plot
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Coverage: target vs empirical
axes[0].plot(1 - cov_df['alpha'], 1 - cov_df['alpha'], 'k--', linewidth=1.2,
             label='Perfect coverage (target=empirical)')
axes[0].plot(1 - cov_df['alpha'], cov_df['empirical_coverage'],
             'o-', color='steelblue', linewidth=2, markersize=8, label='Empirical coverage')
axes[0].fill_between(1 - cov_df['alpha'], 1 - cov_df['alpha'],
                     cov_df['empirical_coverage'], alpha=0.15, color='steelblue')
axes[0].set_xlabel('Target Coverage (1 - α)')
axes[0].set_ylabel('Empirical Coverage')
axes[0].set_title('Coverage Guarantee Verification', fontsize=11, fontweight='bold')
axes[0].legend()
axes[0].set_xlim(0.75, 1.0); axes[0].set_ylim(0.75, 1.0)
sns.despine(ax=axes[0])

# Mean prediction set size vs alpha
axes[1].plot(1 - cov_df['alpha'], cov_df['mean_set_size'],
             's-', color='crimson', linewidth=2, markersize=8)
axes[1].set_xlabel('Target Coverage (1 - α)')
axes[1].set_ylabel('Mean Prediction Set Size')
axes[1].set_title('Set Size vs Coverage Trade-off', fontsize=11, fontweight='bold')
axes[1].axhline(1.0, color='gray', linewidth=1, linestyle=':', label='Singleton (certain)')
axes[1].axhline(4.0, color='gray', linewidth=1, linestyle='--', label='Full set (no info)')
axes[1].legend()
sns.despine(ax=axes[1])

plt.suptitle('Conformal Prediction — Coverage Analysis', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(f'{UNCERT_DIR}/coverage_analysis.png', dpi=150, bbox_inches='tight')
plt.show()


## 4. Prediction Set Size Analysis (at α = 0.10, 90% coverage)

In [ ]:
ALPHA_MAIN = 0.10

y_pred_point_main, pred_sets_raw = mapie_90.predict_set(X_test, alpha=alpha)
pred_sets_main = pred_sets_raw[:, :, 0]
set_sizes_main = pred_sets_main.sum(axis=1)

# Prediction set size distribution
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

vc_size = pd.Series(set_sizes_main).value_counts().sort_index()
axes[0].bar(vc_size.index.astype(str), vc_size.values,
            color=['#2ca02c','#ffbb78','#ff7f0e','#d62728'][:len(vc_size)],
            edgecolor='white', linewidth=0.8)
for i, v in enumerate(vc_size.values):
    axes[0].text(i, v + 30, f'{v:,}\n({v/len(y_test)*100:.1f}%)', ha='center', fontsize=9)
axes[0].set_xlabel('Prediction Set Size')
axes[0].set_ylabel('Count')
axes[0].set_title(f'Prediction Set Size Distribution\n(α={ALPHA_MAIN}, {1-ALPHA_MAIN:.0%} coverage)',
                  fontsize=11, fontweight='bold')
axes[0].set_xticks(range(len(vc_size)))
axes[0].set_xticklabels([f'Size {s}' for s in vc_size.index])
sns.despine(ax=axes[0])

# Set size vs actual class
set_size_by_class = pd.DataFrame({
    'true_class': [LABEL_MAP[c] for c in y_test],
    'set_size':   set_sizes_main
})
order = ['NONE','MINR','SERS','FATL']
palette = {'NONE':'#2ca02c','MINR':'#ffbb78','SERS':'#ff7f0e','FATL':'#d62728'}
sns.boxplot(data=set_size_by_class, x='true_class', y='set_size', order=order,
            palette=palette, linewidth=0.8, ax=axes[1])
axes[1].set_title(f'Prediction Set Size by True Class\n(α={ALPHA_MAIN})',
                  fontsize=11, fontweight='bold')
axes[1].set_xlabel('True Injury Class')
axes[1].set_ylabel('Prediction Set Size')
sns.despine(ax=axes[1])

plt.suptitle('Prediction Set Size — α=0.10', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(f'{UNCERT_DIR}/set_size_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'\nAt α={ALPHA_MAIN} (90% coverage):')
for size, count in vc_size.items():
    print(f'  Size {size}: {count:,} samples ({count/len(y_test)*100:.1f}%)')


## 5. When Is the Model Uncertain?

In [ ]:
# Uncertain = prediction set size > 1
uncertain_mask = set_sizes_main > 1

print(f'Uncertain predictions at α=0.10: {uncertain_mask.sum():,} / {len(y_test):,} '
      f'({uncertain_mask.mean()*100:.1f}%)')

# Load raw test data for feature analysis
try:
    X_test_raw_df = pd.read_csv(f'{OUTPUT_DIR}/X_test.csv')
except:
    X_test_raw_df = None

if X_test_raw_df is not None:
    uncertainty_df = X_test_raw_df.copy()
    uncertainty_df['true_class']  = [LABEL_MAP[c] for c in y_test.values]
    uncertainty_df['set_size']    = set_sizes_main
    uncertainty_df['uncertain']   = uncertain_mask
    uncertainty_df['point_pred']  = [LABEL_MAP[c] for c in y_pred_point_main]
    uncertainty_df['correct_point'] = (uncertainty_df['true_class'] ==
                                        uncertainty_df['point_pred'])

    print('\nUncertain sample breakdown by true class:')
    print(uncertainty_df.groupby('true_class')['uncertain'].agg(['sum','mean']).round(3))


In [ ]:
# Compare mean feature values: certain vs uncertain samples
if X_test_raw_df is not None:
    numeric_features = X_test_raw_df.select_dtypes(include=[np.number]).columns.tolist()

    certain_means   = uncertainty_df.loc[~uncertain_mask, numeric_features].mean()
    uncertain_means = uncertainty_df.loc[uncertain_mask,  numeric_features].mean()

    diff = ((uncertain_means - certain_means) / (certain_means.abs() + 1e-6)).abs()
    diff = diff.sort_values(ascending=False).head(20)

    fig, ax = plt.subplots(figsize=(9, 8))
    diff.sort_values().plot(kind='barh', ax=ax, color='steelblue', edgecolor='white')
    ax.set_title('Features with Largest Relative Difference\nUncertain vs Certain Samples',
                 fontsize=11, fontweight='bold')
    ax.set_xlabel('|Relative mean difference|')
    sns.despine(ax=ax)
    plt.tight_layout()
    plt.savefig(f'{UNCERT_DIR}/uncertain_vs_certain_features.png', dpi=150, bbox_inches='tight')
    plt.show()


In [ ]:
# Weather conditions × uncertainty
if X_test_raw_df is not None and 'noaa_wind_knots' in X_test_raw_df.columns:
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))

    weather_vars = ['noaa_wind_knots', 'noaa_visib_km', 'noaa_temp_c']
    weather_labels = ['Wind Speed (knots)', 'Visibility (km)', 'Temperature (°C)']
    colors_uc = {False:'#2ca02c', True:'#d62728'}

    for ax, var, label in zip(axes, weather_vars, weather_labels):
        if var not in uncertainty_df.columns:
            ax.set_visible(False); continue
        for is_uncert, grp in uncertainty_df.groupby('uncertain'):
            ax.hist(grp[var].dropna(), bins=40, alpha=0.6,
                    color=colors_uc[is_uncert],
                    label='Uncertain' if is_uncert else 'Certain',
                    density=True, edgecolor='white', linewidth=0.3)
        ax.set_title(f'{label}', fontsize=10, fontweight='bold')
        ax.set_xlabel(label)
        ax.set_ylabel('Density')
        ax.legend()
        sns.despine(ax=ax)

    plt.suptitle('Weather Conditions: Certain vs Uncertain Predictions',
                 fontsize=12, fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.savefig(f'{UNCERT_DIR}/weather_vs_uncertainty.png', dpi=150, bbox_inches='tight')
    plt.show()


## 6. Distance to NOAA Station vs Uncertainty

In [ ]:
if X_test_raw_df is not None and 'noaa_dist_km' in X_test_raw_df.columns:
    uncertainty_df['noaa_dist_km'] = X_test_raw_df['noaa_dist_km'].values

    # Bin by distance quartiles
    uncertainty_df['dist_bin'] = pd.qcut(
        uncertainty_df['noaa_dist_km'], q=4,
        labels=['Q1 (closest)','Q2','Q3','Q4 (farthest)'])

    dist_uncert = uncertainty_df.groupby('dist_bin')['uncertain'].agg(['mean','sum','count'])
    dist_uncert['uncertain_pct'] = dist_uncert['mean'] * 100
    print('Uncertainty rate by distance to NOAA station:')
    print(dist_uncert[['uncertain_pct','sum','count']].round(2))

    fig, ax = plt.subplots(figsize=(9, 5))
    dist_uncert['uncertain_pct'].plot(kind='bar', ax=ax, color='steelblue',
                                       edgecolor='white', linewidth=0.8)
    ax.set_title('Uncertain Prediction Rate by Distance to NOAA Station',
                 fontsize=11, fontweight='bold')
    ax.set_ylabel('Uncertain Rate (%)')
    ax.set_xlabel('Distance Quartile')
    ax.set_xticklabels(dist_uncert.index, rotation=30, ha='right')
    sns.despine(ax=ax)
    plt.tight_layout()
    plt.savefig(f'{UNCERT_DIR}/distance_vs_uncertainty.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('noaa_dist_km not found in test features — skipping distance analysis')


## 7. FATL-Specific Coverage Analysis

In [ ]:
# For safety-critical decisions: focus on FATL class coverage
print('FATL-class specific analysis at α=0.10:')

fatl_mask_test = (y_test.values == 3)
fatl_pred_sets = pred_sets_main[fatl_mask_test]
fatl_set_sizes = set_sizes_main[fatl_mask_test]

# Is FATL included in the prediction set for actual FATL cases?
fatl_in_set = fatl_pred_sets[:, 3]  # column 3 = FATL class

print(f'  FATL samples in test set        : {fatl_mask_test.sum():,}')
print(f'  FATL correctly in prediction set: {fatl_in_set.sum():,} ({fatl_in_set.mean()*100:.1f}%)')
print(f'  FATL NOT in prediction set      : {(~fatl_in_set).sum():,}  ← missed fatals')
print(f'  Mean set size for FATL samples  : {fatl_set_sizes.mean():.2f}')

# Coverage at each alpha specifically for FATL
print('\nFATL coverage at each α level:')
for alpha in ALPHA_LEVELS:
    m = MapieClassifier(
        estimator=best_model, ,  random_state=SEED)
    m.fit(X_val, y_val)
    _, ps_raw = m.predict_set(X_test, alpha=alpha)
    ps = ps_raw[:, :, 0]
    fatl_cov = ps[fatl_mask_test, 3].mean()
    print(f'  α={alpha:.2f}  FATL coverage={fatl_cov:.3f}  (target={(1-alpha)*100:.0f}%)')


In [ ]:
# Visualise softmax confidence distribution for FATL vs non-FATL
proba = best_model.predict_proba(X_test)
fatl_conf = proba[:, 3]  # P(FATL) for all test samples

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].hist(fatl_conf[fatl_mask_test],  bins=40, alpha=0.7, color='#d62728',
             label='True FATL', density=True, edgecolor='white')
axes[0].hist(fatl_conf[~fatl_mask_test], bins=40, alpha=0.5, color='#2ca02c',
             label='Not FATL', density=True, edgecolor='white')
axes[0].set_title('P(FATL) Distribution by True Class', fontsize=11, fontweight='bold')
axes[0].set_xlabel('Predicted P(FATL)')
axes[0].set_ylabel('Density')
axes[0].legend()
sns.despine(ax=axes[0])

# Calibration: P(FATL) vs actual FATL rate in bins
n_bins = 10
bins = np.linspace(0, 1, n_bins + 1)
bin_centers, bin_rates, bin_counts = [], [], []
for lo, hi in zip(bins[:-1], bins[1:]):
    mask_b = (fatl_conf >= lo) & (fatl_conf < hi)
    if mask_b.sum() > 0:
        bin_centers.append((lo + hi) / 2)
        bin_rates.append(fatl_mask_test[mask_b].mean())
        bin_counts.append(mask_b.sum())

axes[1].plot([0,1],[0,1],'k--', linewidth=1.2, label='Perfect calibration')
axes[1].scatter(bin_centers, bin_rates, s=[c/2 for c in bin_counts],
                color='steelblue', alpha=0.85, zorder=5, label='Empirical rate')
axes[1].set_title('P(FATL) Calibration Plot', fontsize=11, fontweight='bold')
axes[1].set_xlabel('Predicted P(FATL)')
axes[1].set_ylabel('Empirical FATL rate')
axes[1].legend()
axes[1].set_xlim(0,1); axes[1].set_ylim(0,1)
sns.despine(ax=axes[1])

plt.suptitle('FATL Confidence & Calibration', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(f'{UNCERT_DIR}/fatl_confidence_calibration.png', dpi=150, bbox_inches='tight')
plt.show()


## 8. Example Predictions with Uncertainty

In [ ]:
# Show example predictions: certain, uncertain non-fatal, uncertain fatal
print('Example predictions at α=0.10:')
print('─'*70)

y_pred_pt_ex, ps_ex_raw = mapie_90.predict_set(X_test, alpha=alpha)
ps_ex = ps_ex_raw[:, :, 0]

# Find interesting examples
certain_correct   = np.where((set_sizes_main == 1) & (y_pred_point_main == y_test.values))[0]
certain_wrong     = np.where((set_sizes_main == 1) & (y_pred_point_main != y_test.values))[0]
uncertain_fatl    = np.where((set_sizes_main > 1) & (y_test.values == 3))[0]
uncertain_nonfatl = np.where((set_sizes_main > 1) & (y_test.values != 3))[0]

def describe_set(pred_set_row):
    classes_in_set = [LABEL_MAP[i] for i in range(4) if pred_set_row[i]]
    return '{' + ', '.join(classes_in_set) + '}'

example_indices = []
for group, idx_arr, label in [
    ('CERTAIN & CORRECT',    certain_correct,   5),
    ('CERTAIN & WRONG',      certain_wrong,     5),
    ('UNCERTAIN (true FATL)',uncertain_fatl,    5),
    ('UNCERTAIN (non-FATL)', uncertain_nonfatl, 5),
]:
    print(f'\n[{group}]')
    for i in idx_arr[:label]:
        true  = LABEL_MAP[y_test.values[i]]
        point = LABEL_MAP[y_pred_point_main[i]]
        pset  = describe_set(ps_ex[i])
        conf  = proba[i].max()
        print(f'  idx={i:5d}  true={true}  point_pred={point}  '
              f'pred_set={pset}  max_prob={conf:.3f}')


## 9. Save Conformal Model & Report

In [ ]:
# Save the calibrated conformal predictor
joblib.dump(mapie, f'{UNCERT_DIR}/mapie_classifier_lac.pkl')
print(f'LAC model saved to {UNCERT_DIR}/mapie_classifier_lac.pkl')

# Save coverage results
cov_df.to_csv(f'{UNCERT_DIR}/coverage_results.csv', index=False)
print(f'Coverage table saved')

# Print final summary
print('\n' + '='*65)
print('  CONFORMAL PREDICTION — SUMMARY REPORT')
print('='*65)

best_row = cov_df[cov_df['alpha'] == 0.10].iloc[0]
print(f'  Base model               : {type(best_model).__name__}')
print(f'  CP method                : RAPS (MapieClassifier, cv=prefit)')
print(f'  Calibration set size     : {len(y_val):,}')
print(f'  Test set size            : {len(y_test):,}')
print(f'  ──────────────────────────────────────────────')
print(f'  At α=0.10 (90% target coverage):')
print(f'    Empirical coverage     : {best_row["empirical_coverage"]:.4f}')
print(f'    Mean set size          : {best_row["mean_set_size"]:.3f}')
print(f'    Singleton predictions  : {best_row["frac_singleton"]:.1%}')
print(f'    Full set (no info)     : {best_row["frac_full_set"]:.1%}')
fatl_mask_t = (y_test.values == 3)
fatl_cov_main = pred_sets_main[fatl_mask_t, 3].mean()
print(f'    FATL-class coverage    : {fatl_cov_main:.4f}')
print(f'  ──────────────────────────────────────────────')
print(f'  Coverage guarantee holds: {best_row["empirical_coverage"] >= 1 - 0.10}')
print(f'\n  Artefacts saved to {UNCERT_DIR}/')
for f in sorted(os.listdir(UNCERT_DIR)):
    print(f'    {f}')

print(f'\n  → Next steps:')
print(f'    NB06 (Week 2): SHAP explainability + PredictionPipeline class')
print(f'    NB07 (Week 3): Flask API wrapping full_pipeline.pkl')
